# ALS Baseline: Books + Movies Intersection

Train one implicit ALS model on merged Books and Movies interactions from the prepared Google Drive splits. Recommendations are filtered back to the target domain before MAP@10 evaluation.

In [ ]:
!test -d /content/recommendation-system/.git \
  || git clone -b als https://github.com/mkobycheva/recommendation-system.git /content/recommendation-system

%cd /content/recommendation-system
!git fetch origin als
!git checkout als
!git reset --hard origin/als
!git log -1 --oneline
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

import implicit
import numpy as np
import pandas as pd

from src.data.build_matrix import add_domain_item_ids, build_implicit_als_matrix
from src.evaluation.metrics import map_at_k, ndcg_at_k, recall_at_k

## Load Prepared Splits

The split CSVs are expected to contain the Books/Movies intersecting users and time-based train/validation/test rows.

In [ ]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [ ]:
books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()

## Shared Indexes and Train Matrix

In [ ]:
interaction_matrix = build_implicit_als_matrix(train_df, domains=['books', 'movies'])

user_item_train = interaction_matrix.user_item_train
user2idx = interaction_matrix.user2idx
item2idx = interaction_matrix.item2idx
idx2item = interaction_matrix.idx2item
item_domain = interaction_matrix.item_domain
train_seen_idx_by_user = interaction_matrix.train_seen_idx_by_user
domain_item_indices = interaction_matrix.domain_item_indices

n_users, n_items = user_item_train.shape

print(f'users={n_users:,}, items={n_items:,}, train_interactions={user_item_train.nnz:,}')
print(train_df['domain'].value_counts())

In [ ]:
assert user_item_train.shape == (n_users, n_items)
assert len(domain_item_indices['books']) > 0
assert len(domain_item_indices['movies']) > 0

## Train Collective ALS

In [ ]:
FACTORS = 64
REGULARIZATION = 0.01
ITERATIONS = 20
ALPHA = 40
K = 10

als_model = implicit.als.AlternatingLeastSquares(
    factors=FACTORS,
    regularization=REGULARIZATION,
    iterations=ITERATIONS,
    random_state=42,
)

item_user_train = (user_item_train * ALPHA).T.tocsr()
als_model.fit(item_user_train)

# The model is fit on item-user input, so implicit stores item vectors in
# user_factors and user vectors in item_factors.
als_item_factors = als_model.user_factors
als_user_factors = als_model.item_factors

assert als_item_factors.shape[0] == n_items
assert als_user_factors.shape[0] == n_users

## Domain-Filtered Recommendations

In [ ]:
def relevant_items_by_user(split_df, target_domain):
    domain_rows = split_df[split_df['domain'] == target_domain]
    return domain_rows.groupby('user_id')['item_id'].agg(set).to_dict()


def recommend_for_users(user_ids, target_domain, k=10, batch_size=512, max_score_elements=20_000_000):
    candidates = domain_item_indices[target_domain]
    candidate_factors = als_item_factors[candidates]
    candidate_to_position = {item_idx: pos for pos, item_idx in enumerate(candidates)}
    recommendations = {}
    user_ids = list(user_ids)
    effective_batch_size = max(1, min(batch_size, max_score_elements // max(len(candidates), 1)))

    for batch_start in range(0, len(user_ids), effective_batch_size):
        batch = user_ids[batch_start:batch_start + effective_batch_size]
        valid_user_ids = []
        user_indices = []

        for user_id in batch:
            user_idx = user2idx.get(user_id)
            if user_idx is None:
                recommendations[user_id] = []
                continue
            valid_user_ids.append(user_id)
            user_indices.append(user_idx)

        if not user_indices:
            continue

        user_factors = als_user_factors[np.array(user_indices, dtype=np.int64)]
        all_scores = user_factors @ candidate_factors.T

        for row_idx, user_id in enumerate(valid_user_ids):
            scores = all_scores[row_idx].astype(np.float64, copy=True)
            seen_items = train_seen_idx_by_user.get(user_indices[row_idx], set())
            known_positions = np.array(
                [candidate_to_position[item_idx] for item_idx in seen_items if item_idx in candidate_to_position],
                dtype=np.int64,
            )
            scores[known_positions] = -np.inf

            finite_count = np.isfinite(scores).sum()
            if finite_count == 0:
                recommendations[user_id] = []
                continue

            top_n = min(k, int(finite_count))
            top_positions = np.argpartition(-scores, top_n - 1)[:top_n]
            top_positions = top_positions[np.argsort(-scores[top_positions])]
            recommendations[user_id] = [idx2item[int(candidates[pos])] for pos in top_positions]

        if batch_start % 10000 == 0:
            print(f'Processed {batch_start}/{len(user_ids)} users for {target_domain}...')

    return recommendations

## Smoke Checks

In [ ]:
for domain in ['books', 'movies']:
    sample_relevant_all = relevant_items_by_user(valid_df, domain)
    sample_users = list(sample_relevant_all.keys())[:50]
    sample_recs = recommend_for_users(sample_users, domain, k=K)

    assert all(item.startswith(f'{domain}::') for recs in sample_recs.values() for item in recs)
    for user_id, recs in sample_recs.items():
        user_idx = user2idx[user_id]
        seen_item_ids = {idx2item[idx] for idx in train_seen_idx_by_user[user_idx]}
        assert not (set(recs) & seen_item_ids)

    sample_relevant = {user_id: sample_relevant_all[user_id] for user_id in sample_users}
    assert isinstance(map_at_k(sample_recs, sample_relevant, k=K), float)

## Evaluate and Save Metrics

In [ ]:
def evaluate_multi_domain(split_df, k=10):
    results = {}

    for target_domain in ['books', 'movies']:
        relevant = relevant_items_by_user(split_df, target_domain)
        eval_users = list(relevant.keys())

        print(f'Evaluating {target_domain}: {len(eval_users)} users')
        recs = recommend_for_users(eval_users, target_domain, k=k)

        ndcg_scores = []
        recall_scores = []
        for user_id in eval_users:
            user_recs = recs.get(user_id, [])
            user_relevant = relevant.get(user_id, set())
            if not user_relevant:
                continue
            ndcg_scores.append(ndcg_at_k(user_recs, user_relevant, k))
            recall_scores.append(recall_at_k(user_recs, user_relevant, k))

        results[target_domain] = {
            'ndcg@10': round(float(np.mean(ndcg_scores)), 4) if ndcg_scores else 0.0,
            'recall@10': round(float(np.mean(recall_scores)), 4) if recall_scores else 0.0,
            'map@10': round(float(map_at_k(recs, relevant, k=k)), 4),
            'n_users': len(ndcg_scores),
        }

    return results


K = 10
valid_results = evaluate_multi_domain(valid_df, k=K)
test_results = evaluate_multi_domain(test_df, k=K)

for split_name, split_results in [('valid', valid_results), ('test', test_results)]:
    print(f'\n{split_name}:')
    for domain, domain_metrics in split_results.items():
        print(f"  {domain}:")
        print(f"    NDCG@10:   {domain_metrics['ndcg@10']}")
        print(f"    Recall@10: {domain_metrics['recall@10']}")
        print(f"    MAP@10:    {domain_metrics['map@10']}")
        print(f"    Users:     {domain_metrics['n_users']}")

In [ ]:
os.makedirs('results', exist_ok=True)

result_row = {
    'model': 'implicit_als_books_movies_joint_binary',
    'factors': FACTORS,
    'regularization': REGULARIZATION,
    'iterations': ITERATIONS,
    'alpha': ALPHA,
    'k': K,
    'n_users': n_users,
    'n_items': n_items,
    'n_train_interactions': user_item_train.nnz,
    'books_valid_ndcg@10': valid_results['books']['ndcg@10'],
    'movies_valid_ndcg@10': valid_results['movies']['ndcg@10'],
    'valid_macro_ndcg@10': round((valid_results['books']['ndcg@10'] + valid_results['movies']['ndcg@10']) / 2, 4),
    'books_valid_recall@10': valid_results['books']['recall@10'],
    'movies_valid_recall@10': valid_results['movies']['recall@10'],
    'valid_macro_recall@10': round((valid_results['books']['recall@10'] + valid_results['movies']['recall@10']) / 2, 4),
    'books_valid_map@10': valid_results['books']['map@10'],
    'movies_valid_map@10': valid_results['movies']['map@10'],
    'valid_macro_map@10': round((valid_results['books']['map@10'] + valid_results['movies']['map@10']) / 2, 4),
    'books_test_ndcg@10': test_results['books']['ndcg@10'],
    'movies_test_ndcg@10': test_results['movies']['ndcg@10'],
    'test_macro_ndcg@10': round((test_results['books']['ndcg@10'] + test_results['movies']['ndcg@10']) / 2, 4),
    'books_test_recall@10': test_results['books']['recall@10'],
    'movies_test_recall@10': test_results['movies']['recall@10'],
    'test_macro_recall@10': round((test_results['books']['recall@10'] + test_results['movies']['recall@10']) / 2, 4),
    'books_test_map@10': test_results['books']['map@10'],
    'movies_test_map@10': test_results['movies']['map@10'],
    'test_macro_map@10': round((test_results['books']['map@10'] + test_results['movies']['map@10']) / 2, 4),
}

results_df = pd.DataFrame([result_row])
results_df.to_csv('results/als_baseline_metrics.csv', index=False)
results_df